# 02. Data Cleansing & Feature Engineering

### Project: E-Commerce Customer Funnel & Cohort Retention Analytics

**Objective:** Transform raw transactional logs into a verified, sanitized analytical dataset by executing identifier validation, transaction sanitization (filtering cancellations, inventory returns, and testing artifacts), and deriving core business metrics.

## 1. Environment Setup & Raw Data Ingestion

In [ ]:
import warnings
import pandas as pd

warnings.filterwarnings('ignore')

DATA_PATH = 'data/online_retail_II.xlsx'

print('Ingesting raw partitions...')
df_2009 = pd.read_excel(DATA_PATH, sheet_name='Year 2009-2010', engine='openpyxl')
df_2010 = pd.read_excel(DATA_PATH, sheet_name='Year 2010-2011', engine='openpyxl')
df = pd.concat([df_2009, df_2010], ignore_index=True)

print(f'Raw dataset ingested: {len(df):,} records')

## 2. Identifier Normalization & Customer Validation

Cohort modeling requires tracking customer retention deterministically over time. Records lacking customer IDs represent unregistered guest checkouts and cannot be attributed to cohort life cycles.

In [ ]:
# Standardize identifier column header
df.rename(columns={'Customer ID': 'CustomerID'}, inplace=True)

# Exclude anonymous transactions
initial_len = len(df)
df = df.dropna(subset=['CustomerID'])
print(f'Filtered {initial_len - len(df):,} unassigned guest checkout records.')
print(f'Retained {len(df):,} customer-attributed records.')

## 3. Transaction Sanitization: Cancellations, Reversals & Artifacts

We isolate legitimate commercial purchases by applying three integrity filters:
1. **Cancellations:** Invoices with prefix `C` denote voided or cancelled orders.
2. **Inventory Reversals:** Non-positive quantities indicate returns and credit notes.
3. **Zero / Sub-Zero Unit Prices:** Non-commercial adjustments, samples, or recording anomalies.

In [ ]:
pre_clean_len = len(df)

# Exclude order cancellations
df = df[~df['Invoice'].astype(str).str.startswith('C')]

# Exclude inventory returns and non-positive volume
df = df[df['Quantity'] > 0]

# Exclude zero-value and negative price records
df = df[df['Price'] > 0]

# Deduplicate exact recording duplicates
df = df.drop_duplicates()

print(f'Excluded {pre_clean_len - len(df):,} non-commercial, reversal, and duplicate records.')
print(f'Sanitized transactional base: {len(df):,} rows.')

## 4. Schema Type Casting & Feature Engineering

We cast attributes into strict analytical types and engineer transactional metrics:
- Cast `CustomerID` to discrete string format.
- Parse `InvoiceDate` to standard datetime timestamp.
- Compute `TotalRevenue` as line-item gross realized revenue (`Quantity * Price`).
- Extract `InvoiceMonth` period for temporal segmentation.

In [ ]:
# Normalize types
df['CustomerID'] = df['CustomerID'].astype(int).astype(str)
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

# Feature engineering
df['TotalRevenue'] = (df['Quantity'] * df['Price']).round(2)
df['InvoiceMonth'] = df['InvoiceDate'].dt.to_period('M')

df[['Invoice', 'InvoiceDate', 'CustomerID', 'StockCode', 'Quantity', 'Price', 'TotalRevenue', 'InvoiceMonth']].head()

## 5. Validation Audit & Clean Dataset Serialization

In [ ]:
OUTPUT_PATH = 'data/retail_cleaned.csv'
df.to_csv(OUTPUT_PATH, index=False)

print('--- Dataset Quality Audit ---')
print(f'Total Validated Records : {len(df):,}')
print(f'Total Customer Base     : {df["CustomerID"].nunique():,}')
print(f'Total Completed Orders  : {df["Invoice"].nunique():,}')
print(f'Observation Window      : {df["InvoiceDate"].min().date()} to {df["InvoiceDate"].max().date()}')
print(f'Total Realized Revenue  : £{df["TotalRevenue"].sum():,.2f}')
print(f'Serialized output to    : {OUTPUT_PATH}')